In [1]:
import pandas as pd

df = pd.read_csv("usb_merged_final_data.csv")
print(df.shape)
print(df.dtypes)
print(df.head())
print(df.columns.tolist())

(29775, 16)
id                              object
sessionStart                    object
sessionStop                     object
connectorType                   object
energy_consumption(kWh)        float64
authType                        object
authId                           int64
chargerId                        int64
locationId                       int64
duration_hrs                   float64
time_based_util_rate           float64
Month                            int64
Season                          object
day                             object
Carbon Intensity (gCO2/kWh)      int64
Carbon Emissions (gCO2)        float64
dtype: object
                                     id      sessionStart       sessionStop  \
0  1ff7bb21-86f6-4a58-9713-63c9551b0dc3  18/03/2021 17:39  18/03/2021 17:59   
1  4ec4bf07-f359-4199-93d3-ec842a450cfa  18/03/2021 19:00  18/03/2021 19:33   
2  63ef2126-e813-4225-982d-fcd89108f2ba  18/03/2021 20:43  18/03/2021 21:07   
3  707dbb44-cb45-4cea-ab86-dec7c06

In [2]:
df = pd.read_csv("usb_merged_final_data.csv")
df["sessionStart"] = pd.to_datetime(df["sessionStart"], dayfirst=True)
df["sessionStop"] = pd.to_datetime(df["sessionStop"], dayfirst=True)

print("Date range:", df["sessionStart"].min(), "→", df["sessionStart"].max())
print("Sessions per year:\n", df["sessionStart"].dt.year.value_counts().sort_index())

Date range: 2021-03-18 17:39:00 → 2024-07-01 15:20:00
Sessions per year:
 sessionStart
2021     4035
2022     8515
2023    11264
2024     5961
Name: count, dtype: int64


In [4]:
import duckdb

con = duckdb.connect("data/ev_twin.duckdb")

print("=== metering_raw ===")
print(con.execute("SELECT COUNT(*) FROM metering_raw").fetchone())
print(con.execute("SELECT * FROM metering_raw LIMIT 3").df())

print("=== charging_sessions ===")
print(con.execute("SELECT COUNT(*) FROM charging_sessions").fetchone())
print(con.execute("SELECT * FROM charging_sessions LIMIT 3").df())

print("=== checkpoint ===")
print(con.execute("SELECT status, COUNT(*) FROM scrape_checkpoint GROUP BY status").df())

con.close()

=== metering_raw ===
(28728144,)
      meter_id            datetime  value
0  GM/C/030/01 2021-03-18 00:30:00   9.73
1  GM/C/030/01 2021-03-18 05:00:00   9.73
2  GM/C/030/01 2021-03-18 14:00:00   9.73
=== charging_sessions ===
(29775,)
                                     id       session_start  \
0  1ff7bb21-86f6-4a58-9713-63c9551b0dc3 2021-03-18 17:39:00   
1  4ec4bf07-f359-4199-93d3-ec842a450cfa 2021-03-18 19:00:00   
2  63ef2126-e813-4225-982d-fcd89108f2ba 2021-03-18 20:43:00   

         session_stop      connector_type  energy_kwh auth_type  auth_id  \
0 2021-03-18 17:59:00  IEC_62196_T2_COMBO       9.509  CUSTOMER  1062232   
1 2021-03-18 19:33:00             CHADEMO       6.566  CUSTOMER  1056758   
2 2021-03-18 21:07:00             CHADEMO      13.882  CUSTOMER  1076569   

   charger_id  location_id  duration_hrs  time_based_util_rate  month  season  \
0     5000198        50112      0.325000              1.354167      3  Spring   
1     5000194        50112      0.557500    

In [3]:
con = duckdb.connect("../data/ev_twin.duckdb")

# All 30-min metering windows (both meters summed)
metering = con.execute("""
    SELECT
        datetime AS window_start,
        SUM(value) AS meter_kwh
    FROM metering_raw
    WHERE meter_id IN ('USB_E_SwitchMA', 'USB_E_SwitchMB')
    GROUP BY datetime
    ORDER BY datetime
""").df()

# All 30-min windows from sessions — including zeros for empty windows
sessions = con.execute("""
    WITH windows AS (
        SELECT DISTINCT datetime AS window_start
        FROM metering_raw
        WHERE meter_id = 'USB_E_SwitchMA'
    )
    SELECT
        w.window_start,
        COALESCE(SUM(s.energy_kwh), 0) AS session_kwh,
        COUNT(s.id)                     AS n_sessions
    FROM windows w
    LEFT JOIN charging_sessions s
        ON DATE_TRUNC('hour', s.session_start) +
           INTERVAL (FLOOR(DATEPART('minute', s.session_start) / 30) * 30) MINUTE
           = w.window_start
    GROUP BY w.window_start
    ORDER BY w.window_start
""").df()

merged = sessions.merge(metering, on="window_start", how="inner")
print(f"Total windows : {len(merged)}")
print(f"Windows with sessions: {(merged['n_sessions'] > 0).sum()}")
print(f"Windows without sessions: {(merged['n_sessions'] == 0).sum()}")
print(f"\nCorrelation:\n{merged[['session_kwh','meter_kwh']].corr()}")

# Check overnight baseline
print("\nOvernight sample (no sessions expected):")
print(merged[merged['window_start'].dt.hour < 6][['window_start','session_kwh','meter_kwh']].head(10))

# Check peak hours
print("\nPeak hours sample (10:00-16:00):")
peak = merged[(merged['window_start'].dt.hour >= 10) & 
              (merged['window_start'].dt.hour <= 16)]
print(peak[['window_start','session_kwh','meter_kwh']].head(10))

con.close()

Total windows : 57696
Windows with sessions: 20429
Windows without sessions: 37267

Correlation:
             session_kwh  meter_kwh
session_kwh     1.000000  -0.085573
meter_kwh      -0.085573   1.000000

Overnight sample (no sessions expected):
         window_start  session_kwh  meter_kwh
0 2021-03-18 00:30:00          0.0      90.52
1 2021-03-18 01:00:00          0.0      90.52
2 2021-03-18 01:30:00          0.0      90.52
3 2021-03-18 02:00:00          0.0      90.52
4 2021-03-18 02:30:00          0.0      90.52
5 2021-03-18 03:00:00          0.0      90.52
6 2021-03-18 03:30:00          0.0      90.52
7 2021-03-18 04:00:00          0.0      90.52
8 2021-03-18 04:30:00          0.0      90.52
9 2021-03-18 05:00:00          0.0      90.52

Peak hours sample (10:00-16:00):
          window_start  session_kwh  meter_kwh
19 2021-03-18 10:00:00          0.0      90.52
20 2021-03-18 10:30:00          0.0      90.52
21 2021-03-18 11:00:00          0.0      90.52
22 2021-03-18 11:30:00   

In [2]:
con = duckdb.connect("../data/ev_twin.duckdb")

print(con.execute("""
    SELECT
        meter_id,
        COUNT(*)            AS rows,
        MIN(value)          AS min_val,
        MAX(value)          AS max_val,
        AVG(value)          AS avg_val,
        COUNT(DISTINCT value) AS distinct_values
    FROM metering_raw
    WHERE meter_id IN ('USB_E_SwitchMA', 'USB_E_SwitchMB')
    GROUP BY meter_id
""").df())

# Also check a sample of raw values
print(con.execute("""
    SELECT meter_id, datetime, value
    FROM metering_raw
    WHERE meter_id IN ('USB_E_SwitchMA', 'USB_E_SwitchMB')
    ORDER BY datetime
    LIMIT 20
""").df())

con.close()

         meter_id   rows  min_val  max_val    avg_val  distinct_values
0  USB_E_SwitchMB  57696      0.0   201.85  74.521831              387
1  USB_E_SwitchMA  57696      0.0    21.32  10.508698              454
          meter_id            datetime  value
0   USB_E_SwitchMA 2021-03-18 00:30:00  11.70
1   USB_E_SwitchMB 2021-03-18 00:30:00  78.82
2   USB_E_SwitchMA 2021-03-18 01:00:00  11.70
3   USB_E_SwitchMB 2021-03-18 01:00:00  78.82
4   USB_E_SwitchMA 2021-03-18 01:30:00  11.70
5   USB_E_SwitchMB 2021-03-18 01:30:00  78.82
6   USB_E_SwitchMA 2021-03-18 02:00:00  11.70
7   USB_E_SwitchMB 2021-03-18 02:00:00  78.82
8   USB_E_SwitchMA 2021-03-18 02:30:00  11.70
9   USB_E_SwitchMB 2021-03-18 02:30:00  78.82
10  USB_E_SwitchMA 2021-03-18 03:00:00  11.70
11  USB_E_SwitchMB 2021-03-18 03:00:00  78.82
12  USB_E_SwitchMA 2021-03-18 03:30:00  11.70
13  USB_E_SwitchMB 2021-03-18 03:30:00  78.82
14  USB_E_SwitchMA 2021-03-18 04:00:00  11.70
15  USB_E_SwitchMB 2021-03-18 04:00:00  78.82
16  U

In [4]:
con = duckdb.connect("../data/ev_twin.duckdb")

print(con.execute("""
    SELECT
        DATE_TRUNC('month', datetime) AS month,
        meter_id,
        COUNT(DISTINCT value)         AS distinct_values_per_month,
        MIN(value)                    AS min_val,
        MAX(value)                    AS max_val
    FROM metering_raw
    WHERE meter_id IN ('USB_E_SwitchMA', 'USB_E_SwitchMB')
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df().to_string())

con.close()

        month        meter_id  distinct_values_per_month  min_val  max_val
0  2021-03-01  USB_E_SwitchMA                          1    11.70    11.70
1  2021-03-01  USB_E_SwitchMB                          1    78.82    78.82
2  2021-04-01  USB_E_SwitchMA                          1    11.70    11.70
3  2021-04-01  USB_E_SwitchMB                          1    78.82    78.82
4  2021-05-01  USB_E_SwitchMA                        201     0.00    18.21
5  2021-05-01  USB_E_SwitchMB                         26    49.81   133.30
6  2021-06-01  USB_E_SwitchMA                         76     0.25    18.47
7  2021-06-01  USB_E_SwitchMB                          1    89.74    89.74
8  2021-07-01  USB_E_SwitchMA                          1    12.65    12.65
9  2021-07-01  USB_E_SwitchMB                          1    89.74    89.74
10 2021-08-01  USB_E_SwitchMA                          1    12.65    12.65
11 2021-08-01  USB_E_SwitchMB                          1    89.74    89.74
12 2021-09-01  USB_E_Swit